# Module 06 — Counting-Based Bigram Model

Phase 1 was about *how a network learns*. Phase 2 is about *language
modeling*: given some text, predict what character/word comes next. This
module builds the simplest possible language model — no neural network at
all, just counting — as the baseline everything later in this project
improves on.

**The idea:** for every pair of consecutive characters `(ch1, ch2)` in the
training text, count how often `ch2` follows `ch1`. Normalize those counts
into probabilities, and you have a model: "given the current character,
here's the probability distribution over what comes next."

**The corpus:** a list of Genshin Impact character names — small, fun, and
thematically on-brand for where this project is headed. The model will
learn to generate new, name-*like* strings by picking up on patterns like
"which letters tend to start a name" and "which letters tend to follow
which."

## 1. The corpus and vocabulary

In [ ]:
names = [
    "aether", "lumine", "amber", "kaeya", "lisa", "jean", "barbara", "diluc",
    "noelle", "bennett", "fischl", "sucrose", "chongyun", "klee", "xingqiu",
    "ningguang", "beidou", "xiangling", "xiao", "zhongli", "hutao", "yanfei",
    "rosaria", "albedo", "diona", "mona", "keqing", "qiqi", "venti",
    "tartaglia", "ganyu", "xinyan", "sayu", "kokomi", "kazuha", "ayaka",
    "yoimiya", "sara", "raiden", "aloy", "itto", "gorou", "yaemiko",
    "shinobu", "heizou", "yelan", "tighnari", "nahida", "nilou", "cyno",
    "candace", "layla", "wanderer", "faruzan", "dehya", "mika", "kaveh",
    "baizhu", "kirara", "lynette", "lyney", "freminet", "neuvillette",
    "wriothesley", "charlotte", "furina", "chevreuse", "navia", "chiori",
    "arlecchino", "clorinde", "sigewinne", "emilie", "kachina", "kinich",
    "mualani", "xilonen", "ororon", "chasca", "mavuika", "citlali", "varesa",
    "iansan", "escoffier", "ineffa",
]
print(f"{len(names)} names, e.g. {names[:5]}")

# vocabulary: every character that appears, plus one special token "." used
# as both start-of-name and end-of-name
chars = sorted(set("".join(names)))
vocab = ["."] + chars
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
print(f"vocab size: {len(vocab)}")
print(vocab)

## 2. Counting bigrams

For each name, wrap it in `.` on both ends (`"kaeya"` → `".kaeya."`) so the
model also learns which letters tend to *start* a name (bigram `. -> k`)
and which end one (bigram `a -> .`). Then just count every consecutive
pair.

In [ ]:
import torch

N = torch.zeros((len(vocab), len(vocab)), dtype=torch.int32)

for name in names:
    wrapped = "." + name + "."
    for ch1, ch2 in zip(wrapped, wrapped[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

print("Example: how many times is \'a\' followed by \'e\'?", N[stoi["a"], stoi["e"]].item())
print("Example: how many names start with \'k\'?", N[stoi["."], stoi["k"]].item())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.imshow(N, cmap="Blues")
plt.xticks(range(len(vocab)), vocab, fontsize=7)
plt.yticks(range(len(vocab)), vocab, fontsize=7)
plt.xlabel("2nd character")
plt.ylabel("1st character")
plt.title("Bigram counts")
plt.show()

## 3. Counts → probabilities

Normalize each row so it sums to 1 — row `i` becomes "the probability
distribution over what character follows character `i`." We add 1 to every
count first (Laplace/add-one smoothing) so no bigram ever gets probability
exactly 0, which would make the model assign *infinite* loss to any unseen
combination — a real problem once we score the model in the next step.

In [ ]:
P = (N + 1).float()
P = P / P.sum(dim=1, keepdim=True)

print("Probability distribution over what follows \'k\':")
for ch, p in zip(vocab, P[stoi["k"]].tolist()):
    if p > 0.02:
        print(f"  {ch!r}: {p:.3f}")

## 4. Sampling new, name-like strings

In [ ]:
generator = torch.Generator().manual_seed(42)

def sample_name():
    out = []
    ix = stoi["."]
    while True:
        ix = torch.multinomial(P[ix], num_samples=1, generator=generator).item()
        if ix == stoi["."]:
            break
        out.append(itos[ix])
    return "".join(out)


for _ in range(10):
    print(sample_name())

## 5. Scoring the model: negative log-likelihood

Generated strings *look* plausible, but "looks plausible to a human" isn't
a number we can optimize or compare across models. The standard way to
score a language model is: for every bigram actually seen in the training
data, look up the probability the model assigned to it, and average the
negative log of those probabilities. Lower is better — a perfect model that
always predicted the true next character with probability 1 would score
exactly 0.

This exact quantity, exponentiated, is called **perplexity** — the subject
of Module 07. Here we just compute the raw average negative log-likelihood
to confirm the counting model learned something non-trivial.

In [ ]:
log_likelihood = 0.0
n = 0
for name in names:
    wrapped = "." + name + "."
    for ch1, ch2 in zip(wrapped, wrapped[1:]):
        prob = P[stoi[ch1], stoi[ch2]]
        log_likelihood += torch.log(prob).item()
        n += 1

nll = -log_likelihood / n
print(f"Average negative log-likelihood: {nll:.4f}")

# Sanity check: a uniform-random model (every next-char equally likely)
# should score noticeably worse than our counting model.
uniform_nll = -torch.log(torch.tensor(1.0 / len(vocab))).item()
print(f"Uniform-random baseline:         {uniform_nll:.4f}")
assert nll < uniform_nll, "the counting model should beat random guessing"
print("\nThe counting model assigns meaningfully higher probability to real bigrams than chance would.")

## Recap

- A language model is just something that outputs a probability
  distribution over "what comes next," given what came before.
- The absolute simplest version needs no learning algorithm at all —
  just counting frequencies and normalizing.
- We can already generate plausible-*looking* new strings and score the
  model's quality with a single number (average negative log-likelihood).
- What's missing: this model only ever looks at **one** previous
  character. Module 09's neural n-gram model looks at a fixed window of
  several; the Transformer (Phase 3) eventually looks at everything before
  it. Next up — Module 07: turning that raw NLL number into perplexity, the
  standard way language models get evaluated and compared.